In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder,StandardScaler,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score,roc_auc_score

In [2]:
df = pd.read_csv('data/WA_Fn-UseC_-Telco-Customer-Churn.xls')

In [3]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [5]:
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].mean())

In [6]:
df['HasFamily'] = ((df['Partner'] == 'Yes') | (df['Dependents'] == 'Yes')).map({True: 'Yes', False: 'No'})

In [7]:
df.drop(columns=['customerID','Partner','Dependents'],inplace=True)

In [8]:
X = df.drop(columns=['Churn'])
y = df['Churn']
X

,gender,SeniorCitizen,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,HasFamily
0,Female,0,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,Yes
1,Male,0,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,No
3,Male,0,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,Yes
7039,Female,0,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,Yes
7040,Female,0,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,Yes
7041,Male,1,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,Yes


In [9]:
numerical_feature = X.select_dtypes(exclude="str").columns
categorical_feature = X.select_dtypes(include="str").columns
categorical_feature

Index(['gender', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'HasFamily'],
      dtype='str')

In [10]:
for col in categorical_feature:
    df[col] =  df[col].str.replace(' ', '')
    df[col] =  df[col].str.replace('-', '')

In [11]:
X.head()

,gender,SeniorCitizen,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,HasFamily
0,Female,0,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,Yes
1,Male,0,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,No
3,Male,0,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,No


In [12]:
preprocessor = ColumnTransformer([
    ('OHE',OneHotEncoder(drop='first'),categorical_feature),
    ('SC',StandardScaler(),numerical_feature)
])

In [13]:
le = LabelEncoder()

y = le.fit_transform(y)
y

array([0, 0, 1, ..., 0, 1, 0], shape=(7043,))

In [14]:
X = preprocessor.fit_transform(X)


In [15]:
X

array([[ 0.        ,  0.        ,  1.        , ..., -1.27744458,
        -1.16032292, -0.99497138],
       [ 1.        ,  1.        ,  0.        , ...,  0.06632742,
        -0.25962894, -0.17387565],
       [ 1.        ,  1.        ,  0.        , ..., -1.23672422,
        -0.36266036, -0.96039939],
       ...,
       [ 0.        ,  0.        ,  1.        , ..., -0.87024095,
        -1.1686319 , -0.85518222],
       [ 1.        ,  1.        ,  0.        , ..., -1.15528349,
         0.32033821, -0.87277729],
       [ 1.        ,  1.        ,  0.        , ...,  1.36937906,
         1.35896134,  2.01391739]], shape=(7043, 29))

In [16]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)
print(X_train)
print(y_train)

[[ 1.          1.          0.         ...  0.88073469  0.19736523
   0.65642602]
 [ 1.          1.          0.         ... -1.27744458  0.52473924
  -0.97258569]
 [ 1.          1.          0.         ... -0.78880022 -1.51096208
  -0.89350724]
 ...
 [ 1.          1.          0.         ... -0.82952058 -1.44947559
  -0.87302013]
 [ 1.          1.          0.         ... -0.82952058  1.15289851
  -0.47824601]
 [ 1.          1.          0.         ... -0.25943549 -1.49434411
  -0.80623836]]
[0 0 0 ... 0 1 0]


In [17]:
models = {
                "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
                "Random Forest": RandomForestClassifier(class_weight='balanced', n_estimators=200, random_state=42),
                "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
                "LightGBM": LGBMClassifier(class_weight='balanced', random_state=42),
                "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
                "Gradient Boosting": GradientBoostingClassifier(random_state=42),
                "SVM": SVC(class_weight='balanced', probability=True, random_state=42),
                "KNN": KNeighborsClassifier(n_neighbors=5)
            }

In [18]:
report = {}
for model_name,model in models.items():

    model.fit(X_train,y_train)

    predict = model.predict(X_test)

    roc_auc = roc_auc_score(y_test,predict)

    report[model_name] = roc_auc

[LightGBM] [Info] Number of positive: 1295, number of negative: 3635
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001792 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 637
[LightGBM] [Info] Number of data points in the train set: 4930, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


c:\Users\Dhvanish\OneDrive\Desktop\ML projects\Customer churn prediction\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [19]:
for model, score in sorted(report.items(), key=lambda item: item[1], reverse=True):
    print(f"{model}: {score:.4f}")


Logistic Regression: 0.7805
SVM: 0.7722
LightGBM: 0.7648
Random Forest: 0.7372
Gradient Boosting: 0.7123
CatBoost: 0.7110
XGBoost: 0.6992
KNN: 0.6919
